In [0]:

# =====================================================
# CAPSTONE 03 - DAY 2
# SILVER LAYER CONFIGURATION
# =====================================================

CATALOG = "capstone_3flight"
BRONZE_SCHEMA = "flight_schema"
SILVER_SCHEMA = "silver_schema"

# Bronze source tables
BRONZE_FLIGHT_TABLE = f"{CATALOG}.{BRONZE_SCHEMA}.bronze_flight_events"
BRONZE_CDC_TABLE = f"{CATALOG}.{BRONZE_SCHEMA}.bronze_status_cdc"
BRONZE_SECONDARY_TABLE = f"{CATALOG}.{BRONZE_SCHEMA}.bronze_secondary_events"

# Reference tables
AIRLINE_TABLE = f"{CATALOG}.{BRONZE_SCHEMA}.airline"
AIRPORT_TABLE = f"{CATALOG}.{BRONZE_SCHEMA}.airport"

# Silver target tables
SILVER_FLIGHT_TABLE = f"{CATALOG}.{SILVER_SCHEMA}.silver_flight_events"
SILVER_REJECTED_TABLE = f"{CATALOG}.{SILVER_SCHEMA}.silver_rejected_records"
SILVER_STATUS_HISTORY = f"{CATALOG}.{SILVER_SCHEMA}.silver_status_history"
SILVER_FLIGHT_CURRENT = f"{CATALOG}.{SILVER_SCHEMA}.silver_flight_current"

# Verify schemas and source tables
spark.sql(f"DESCRIBE SCHEMA {CATALOG}.{SILVER_SCHEMA}")

for table in [
    BRONZE_FLIGHT_TABLE,
    BRONZE_CDC_TABLE,
    BRONZE_SECONDARY_TABLE,
    AIRLINE_TABLE,
    AIRPORT_TABLE
]:
    print(table, ":", spark.table(table).count(), "rows")

print("Silver configuration verified.")


In [0]:

# =====================================================
# DATA QUALITY RULE CATALOGUE
# =====================================================

dq_rules = [
    (
        "DQ01",
        "flight_id",
        "Must not be NULL or empty",
        "CRITICAL",
        "QUARANTINE",
        "Every flight event must have a business identifier"
    ),
    (
        "DQ02",
        "event_timestamp",
        "Must be a valid timestamp",
        "CRITICAL",
        "QUARANTINE",
        "Flight event time is mandatory"
    ),
    (
        "DQ03",
        "fare_amount",
        "Must be a valid numeric value",
        "HIGH",
        "QUARANTINE",
        "Fare must be usable for downstream analysis"
    ),
    (
        "DQ04",
        "flight_type",
        "Optional field; NULL is permitted",
        "MEDIUM",
        "FLAG",
        "Missing optional values must be tracked"
    ),
    (
        "DQ05",
        "fare_amount",
        "Identify project-injected extreme values",
        "MEDIUM",
        "FLAG",
        "Extreme fare threshold requires documented approval"
    ),
    (
        "DQ06",
        "airline_id",
        "Must match airline reference data",
        "HIGH",
        "QUARANTINE",
        "Prevent unexplained airline reference mismatches"
    ),
    (
        "DQ07",
        "airport_id",
        "Must match airport reference data",
        "HIGH",
        "QUARANTINE",
        "Prevent unexplained airport reference mismatches"
    ),
    (
        "DQ08",
        "flight_id",
        "Check repeated source business keys",
        "MEDIUM",
        "DEDUPLICATE",
        "Apply deterministic deduplication in Silver"
    )
]

dq_columns = [
    "rule_id",
    "column_name",
    "validation_condition",
    "severity",
    "expected_action",
    "description"
]

dq_df = spark.createDataFrame(dq_rules, dq_columns)

display(dq_df)

print("Total DQ rules:", dq_df.count())


In [0]:
DQ_TABLE = f"{CATALOG}.{SILVER_SCHEMA}.dq_rule_catalogue"

(
    dq_df.write
    .format("delta")
    .mode("overwrite")
    .saveAsTable(DQ_TABLE)
)

print("DQ catalogue saved successfully:", DQ_TABLE)

In [0]:
saved_dq = spark.table(DQ_TABLE)

print("Saved DQ rules:", saved_dq.count())

display(saved_dq)

In [0]:

# ==========================================
# TASK 4.1 - READ BRONZE FLIGHT EVENTS
# ==========================================

from pyspark.sql import functions as F
from pyspark.sql.window import Window
from pyspark.sql.types import *

# Read the existing Bronze table
flight_bronze_df = spark.table(BRONZE_FLIGHT_TABLE)

# Display total records
print("Total Bronze records:", flight_bronze_df.count())

# Display schema
print("\nBronze schema:")
flight_bronze_df.printSchema()

# Display sample records
print("\nSample flight records:")
display(flight_bronze_df.limit(10))

# Check rescued records
print("\nRescued data count:")

display(
    flight_bronze_df
    .filter(F.col("_rescued_data").isNotNull())
    .count()
    if "_rescued_data" in flight_bronze_df.columns
    else 0
)


In [0]:

# ==========================================
# TASK 4.2 - SILVER DATA QUALITY PROFILING
# ==========================================

# 1. Check missing values

columns_to_check = [
    "flight_id",
    "event_timestamp",
    "ingest_timestamp",
    "airline_id",
    "airport_id",
    "flight_type",
    "flight_status",
    "fare_amount",
    "delay_minutes"
]

null_expressions = [
    F.count(
        F.when(F.col(c).isNull(), 1)
    ).alias(c)
    for c in columns_to_check
]

print("NULL VALUE PROFILE")
display(flight_bronze_df.agg(*null_expressions))


# 2. Check flight status distribution

print("FLIGHT STATUS DISTRIBUTION")

display(
    flight_bronze_df
    .groupBy("flight_status")
    .count()
    .orderBy("flight_status")
)


# 3. Check flight type distribution

print("FLIGHT TYPE DISTRIBUTION")

display(
    flight_bronze_df
    .groupBy("flight_type")
    .count()
    .orderBy("flight_type")
)


# 4. Check fare statistics

print("FARE STATISTICS")

display(
    flight_bronze_df
    .select(
        F.min("fare_amount").alias("minimum_fare"),
        F.max("fare_amount").alias("maximum_fare"),
        F.avg("fare_amount").alias("average_fare"),
        F.expr("percentile_approx(fare_amount, 0.5)")
         .alias("median_fare")
    )
)


# 5. Check airline reference coverage

airline_keys = spark.table(AIRLINE_TABLE).select("airline_id").distinct()

print("AIRLINE KEY COVERAGE")

display(
    flight_bronze_df
    .join(airline_keys, "airline_id", "left_anti")
    .count()
)


# 6. Check airport reference coverage

airport_keys = spark.table(AIRPORT_TABLE).select("airport_id").distinct()

print("AIRPORT KEY COVERAGE")

display(
    flight_bronze_df
    .join(airport_keys, "airport_id", "left_anti")
    .count()
)


In [0]:

# ==========================================
# SOURCE ANOMALY VERIFICATION
# ==========================================

# 1. VERIFY EXACT DUPLICATES
# Use only the original 12 source columns.
# Exclude Bronze ingestion metadata.

source_columns = [
    "flight_id",
    "event_timestamp",
    "ingest_timestamp",
    "airline_id",
    "airport_id",
    "flight_type",
    "flight_status",
    "timezone",
    "fare_amount",
    "delay_minutes",
    "aircraft_id",
    "source_system"
]

duplicate_groups = (
    flight_bronze_df
    .groupBy(*source_columns)
    .count()
    .filter(F.col("count") > 1)
)

duplicate_summary = duplicate_groups.agg(
    F.sum(F.col("count") - 1).alias("extra_duplicate_rows"),
    F.count("*").alias("duplicate_groups")
)

print("EXACT DUPLICATE VERIFICATION")
display(duplicate_summary)


# 2. VERIFY EXTREME FARE VALUES
# Project dataset finding: 35 injected values exceed 5000.

extreme_fares = flight_bronze_df.filter(
    F.col("fare_amount") > 5000
)

print("EXTREME FARE VERIFICATION")
print("Extreme fare records:", extreme_fares.count())

display(
    extreme_fares.select(
        "flight_id",
        "fare_amount",
        "airline_id",
        "airport_id"
    )
)


# 3. VERIFY DISTINCT FLIGHT STATUSES

print("FLIGHT STATUS VERIFICATION")

display(
    flight_bronze_df
    .groupBy("flight_status")
    .count()
    .orderBy("flight_status")
)

status_count = (
    flight_bronze_df
    .select("flight_status")
    .distinct()
    .count()
)

print("Number of distinct flight statuses:", status_count)


In [0]:

# ==========================================
# TASK 4.3 - STANDARDISE FLIGHT EVENTS
# ==========================================

silver_prepared_df = flight_bronze_df

# 1. Preserve original source values
columns_to_preserve = [
    "flight_id",
    "flight_type",
    "flight_status",
    "airline_id",
    "airport_id",
    "fare_amount",
    "delay_minutes",
    "timezone"
]

for column in columns_to_preserve:
    silver_prepared_df = silver_prepared_df.withColumn(
        f"{column}_raw",
        F.col(column)
    )

# 2. Standardise identifiers
for column in ["flight_id", "airline_id", "airport_id", "aircraft_id"]:
    silver_prepared_df = silver_prepared_df.withColumn(
        column,
        F.trim(F.col(column))
    )

# 3. Standardise categorical fields
for column in ["flight_type", "flight_status", "source_system"]:
    silver_prepared_df = silver_prepared_df.withColumn(
        column,
        F.upper(F.trim(F.col(column)))
    )

# 4. Convert empty flight_type strings to NULL
silver_prepared_df = silver_prepared_df.withColumn(
    "flight_type",
    F.when(F.col("flight_type") == "", None)
     .otherwise(F.col("flight_type"))
)

# 5. Ensure expected data types
silver_prepared_df = (
    silver_prepared_df
    .withColumn("event_timestamp", F.col("event_timestamp").cast("timestamp"))
    .withColumn("ingest_timestamp", F.col("ingest_timestamp").cast("timestamp"))
    .withColumn("fare_amount", F.col("fare_amount").cast("double"))
    .withColumn("delay_minutes", F.col("delay_minutes").cast("integer"))
)

# 6. Add missing optional field quality flag
silver_prepared_df = silver_prepared_df.withColumn(
    "dq_missing_flight_type",
    F.col("flight_type").isNull()
)

# 7. Validate prepared records
print("Silver standardisation completed.")
print("Prepared records:", silver_prepared_df.count())

display(silver_prepared_df.limit(10))

print("Prepared schema:")
silver_prepared_df.printSchema()


In [0]:

# Verify standardisation results

print("Prepared record count:", silver_prepared_df.count())

print(
    "Missing flight_type flags:",
    silver_prepared_df
    .filter(F.col("dq_missing_flight_type") == True)
    .count()
)

print(
    "NULL flight_type values:",
    silver_prepared_df
    .filter(F.col("flight_type").isNull())
    .count()
)

display(
    silver_prepared_df
    .groupBy("dq_missing_flight_type")
    .count()
)


In [0]:

# ==========================================
# TASK 4.4 - MANDATORY FIELD VALIDATION
# ==========================================

# Identify invalid mandatory fields

silver_validated_df = (
    silver_prepared_df
    .withColumn(
        "dq_reason",
        F.concat_ws(
            "; ",
            F.when(
                F.col("flight_id").isNull() |
                (F.trim(F.col("flight_id")) == ""),
                F.lit("DQ01: Missing flight_id")
            ),
            F.when(
                F.col("event_timestamp").isNull(),
                F.lit("DQ02: Missing event_timestamp")
            ),
            F.when(
                F.col("fare_amount").isNull(),
                F.lit("DQ03: Missing or invalid fare_amount")
            )
        )
    )
    .withColumn(
        "dq_status",
        F.when(F.col("dq_reason") != "", "REJECTED")
         .otherwise("VALID")
    )
)

# Separate valid and rejected records

silver_valid_candidates = silver_validated_df.filter(
    F.col("dq_status") == "VALID"
)

silver_rejected_candidates = silver_validated_df.filter(
    F.col("dq_status") == "REJECTED"
)

print("Total records:", silver_validated_df.count())
print("Valid candidates:", silver_valid_candidates.count())
print("Rejected candidates:", silver_rejected_candidates.count())

display(
    silver_validated_df.groupBy("dq_status").count()
)


In [0]:

# ==========================================
# TASK 4.5 - REFERENCE INTEGRITY VALIDATION
# ==========================================

airline_keys = (
    spark.table(AIRLINE_TABLE)
    .select("airline_id")
    .distinct()
)

airport_keys = (
    spark.table(AIRPORT_TABLE)
    .select("airport_id")
    .distinct()
)

# Find unmatched airline IDs
unmatched_airlines = (
    silver_valid_candidates
    .join(airline_keys, "airline_id", "left_anti")
)

# Find unmatched airport IDs
unmatched_airports = (
    silver_valid_candidates
    .join(airport_keys, "airport_id", "left_anti")
)

print("Unmatched airline records:", unmatched_airlines.count())
print("Unmatched airport records:", unmatched_airports.count())

# Display unmatched records if any
display(unmatched_airlines)
display(unmatched_airports)


In [0]:

# ==========================================
# TASK 4.6 - FINALISE VALIDATED DATASETS
# ==========================================

silver_validated_final_df = (
    silver_validated_df
    .withColumn(
        "dq_reason",
        F.concat_ws(
            "; ",
            F.when(
                F.col("flight_id").isNull() |
                (F.trim(F.col("flight_id")) == ""),
                F.lit("DQ01: Missing flight_id")
            ),
            F.when(
                F.col("event_timestamp").isNull(),
                F.lit("DQ02: Missing event_timestamp")
            ),
            F.when(
                F.col("fare_amount").isNull(),
                F.lit("DQ03: Missing or invalid fare_amount")
            ),
            F.when(
                ~F.col("airline_id").isin(
                    *[r["airline_id"] for r in airline_keys.collect()]
                ),
                F.lit("DQ06: Unknown airline_id")
            ),
            F.when(
                ~F.col("airport_id").isin(
                    *[r["airport_id"] for r in airport_keys.collect()]
                ),
                F.lit("DQ07: Unknown airport_id")
            )
        )
    )
    .withColumn(
        "dq_status",
        F.when(F.col("dq_reason") != "", "REJECTED")
         .otherwise("VALID")
    )
)

silver_valid_final_df = silver_validated_final_df.filter(
    F.col("dq_status") == "VALID"
)

silver_rejected_final_df = silver_validated_final_df.filter(
    F.col("dq_status") == "REJECTED"
)

print("Total records:", silver_validated_final_df.count())
print("Valid records:", silver_valid_final_df.count())
print("Rejected records:", silver_rejected_final_df.count())


In [0]:
# ==========================================
# TASK 4.7 - SAVE REJECTED RECORDS TABLE
# ==========================================

(
    silver_rejected_final_df
    .write
    .format("delta")
    .mode("overwrite")
    .option("overwriteSchema", "true")
    .saveAsTable(SILVER_REJECTED_TABLE)
)

print("Rejected-records table created:", SILVER_REJECTED_TABLE)
print("Rows saved:", spark.table(SILVER_REJECTED_TABLE).count())
print("Columns:")
spark.table(SILVER_REJECTED_TABLE).printSchema()

Step 5.1 – Measure exact duplicates


In [0]:
# ==========================================
# TASK 5.2 - COMPARE DUPLICATE DEFINITIONS
# ==========================================

source_columns = [
    "flight_id",
    "event_timestamp",
    "ingest_timestamp",
    "airline_id",
    "airport_id",
    "flight_type",
    "flight_status",
    "timezone",
    "fare_amount",
    "delay_minutes",
    "aircraft_id",
    "source_system"
]

# Duplicate rows using original source columns
source_duplicate_groups = (
    silver_valid_final_df
    .groupBy(*source_columns)
    .count()
    .filter(F.col("count") > 1)
)

source_duplicate_rows = (
    source_duplicate_groups
    .select(F.sum(F.col("count") - 1).alias("n"))
    .first()["n"]
)

# Compare against the existing business-column result
print("Duplicate rows using original source columns:",
      source_duplicate_rows or 0)

print("Duplicate rows using business columns:",
      duplicate_rows_to_remove or 0)

print("Original Bronze row count:",
      flight_bronze_df.count())

print("Current valid DataFrame row count:",
      silver_valid_final_df.count())

In [0]:
# ==========================================
# TASK 5.3 - REMOVE EXACT SOURCE DUPLICATES
# ==========================================

from pyspark.sql.window import Window

source_columns = [
    "flight_id",
    "event_timestamp",
    "ingest_timestamp",
    "airline_id",
    "airport_id",
    "flight_type",
    "flight_status",
    "timezone",
    "fare_amount",
    "delay_minutes",
    "aircraft_id",
    "source_system"
]

# Assign a deterministic row number within each exact source group.
# The selected source columns are identical within each group.
dedup_window = (
    Window
    .partitionBy(*source_columns)
    .orderBy(F.col("_source_file").asc(), F.col("_ingested_at").asc())
)

silver_deduplicated_df = (
    silver_valid_final_df
    .withColumn("_duplicate_rank", F.row_number().over(dedup_window))
    .filter(F.col("_duplicate_rank") == 1)
    .drop("_duplicate_rank")
)

print("Before deduplication:", silver_valid_final_df.count())
print("After deduplication:", silver_deduplicated_df.count())
print(
    "Duplicate rows removed:",
    silver_valid_final_df.count() - silver_deduplicated_df.count()
)

Step 5.4 – Save the deduplicated Silver flight table

In [0]:
# ==========================================
# TASK 5.4 - SAVE SILVER FLIGHT EVENTS
# ==========================================

(
    silver_deduplicated_df
    .write
    .format("delta")
    .mode("overwrite")
    .option("overwriteSchema", "true")
    .saveAsTable(SILVER_FLIGHT_TABLE)
)

saved_silver_df = spark.table(SILVER_FLIGHT_TABLE)

print("Silver table created:", SILVER_FLIGHT_TABLE)
print("Rows saved:", saved_silver_df.count())

display(
    saved_silver_df.select(
        "flight_id",
        "event_timestamp",
        "airline_id",
        "airport_id",
        "flight_status",
        "fare_amount",
        "dq_missing_flight_type"
    ).limit(10)
)

In [0]:
# ==========================================
# TASK 6.1 - INSPECT BRONZE STATUS CDC
# ==========================================

cdc_df = spark.table(BRONZE_CDC_TABLE)

print("Total CDC records:", cdc_df.count())

cdc_df.printSchema()

display(
    cdc_df.orderBy(
        F.col("flight_id"),
        F.col("source_sequence")
    ).limit(15)
)

In [0]:
# ==========================================
# TASK 6.2 - PROFILE CDC STATUS UPDATES
# ==========================================

from pyspark.sql import functions as F

# 1. Check missing mandatory CDC fields
cdc_df.select(
    F.count("*").alias("total_records"),
    F.sum(F.col("flight_id").isNull().cast("int")).alias("null_flight_id"),
    F.sum(F.col("change_timestamp").isNull().cast("int")).alias("null_change_timestamp"),
    F.sum(F.col("source_sequence").isNull().cast("int")).alias("null_source_sequence"),
    F.sum(F.col("new_status").isNull().cast("int")).alias("null_new_status")
).show(truncate=False)

# 2. Check sequence range
cdc_df.select(
    F.min("source_sequence").alias("min_sequence"),
    F.max("source_sequence").alias("max_sequence"),
    F.countDistinct("source_sequence").alias("distinct_sequences")
).show()

# 3. Check whether a flight has repeated sequence numbers
repeated_sequences = (
    cdc_df
    .groupBy("flight_id", "source_sequence")
    .count()
    .filter(F.col("count") > 1)
)

print("Repeated flight/sequence combinations:", repeated_sequences.count())

# 4. Inspect status transitions in sequence order
display(
    cdc_df
    .orderBy("flight_id", "source_sequence", "change_timestamp")
    .select(
        "flight_id",
        "source_sequence",
        "change_timestamp",
        "previous_status",
        "new_status",
        "change_reason"
    )
    .limit(20)
)

# 5. Count new statuses
display(
    cdc_df
    .groupBy("new_status")
    .count()
    .orderBy("new_status")
)

In [0]:
# ==========================================
# TASK 6.3 - BUILD SILVER STATUS HISTORY
# ==========================================

silver_status_history_df = (
    cdc_df
    .withColumn(
        "previous_status",
        F.upper(F.trim(F.col("previous_status")))
    )
    .withColumn(
        "new_status",
        F.upper(F.trim(F.col("new_status")))
    )
    .withColumn(
        "change_reason",
        F.trim(F.col("change_reason"))
    )
    .withColumn(
        "processed_at",
        F.current_timestamp()
    )
)

print("Status history records:", silver_status_history_df.count())

display(
    silver_status_history_df
    .orderBy("source_sequence")
    .select(
        "flight_id",
        "source_sequence",
        "change_timestamp",
        "previous_status",
        "new_status",
        "change_reason"
    )
    .limit(15)
)

In [0]:
# ==========================================
# TASK 6.4 - SAVE SILVER STATUS HISTORY
# ==========================================

(
    silver_status_history_df
    .write
    .format("delta")
    .mode("overwrite")
    .option("overwriteSchema", "true")
    .saveAsTable(SILVER_STATUS_HISTORY)
)

saved_history_df = spark.table(SILVER_STATUS_HISTORY)

print("Status history table:", SILVER_STATUS_HISTORY)
print("Rows saved:", saved_history_df.count())

display(
    saved_history_df
    .orderBy("source_sequence")
    .limit(10)
)

In [0]:
# ==========================================
# TASK 6.5 - BUILD CURRENT FLIGHT STATUS
# ==========================================

from pyspark.sql.window import Window

latest_status_window = (
    Window
    .partitionBy("flight_id")
    .orderBy(
        F.col("source_sequence").desc(),
        F.col("change_timestamp").desc()
    )
)

silver_flight_current_df = (
    silver_status_history_df
    .withColumn(
        "_status_rank",
        F.row_number().over(latest_status_window)
    )
    .filter(F.col("_status_rank") == 1)
    .drop("_status_rank")
)

print("Total CDC records:", silver_status_history_df.count())
print("Distinct flights with status updates:",
      silver_flight_current_df.count())

display(
    silver_flight_current_df
    .select(
        "flight_id",
        "source_sequence",
        "change_timestamp",
        "previous_status",
        "new_status",
        "change_reason"
    )
    .orderBy("flight_id")
    .limit(15)
)

In [0]:
# ==========================================
# TASK 6.6 - SAVE SILVER FLIGHT CURRENT
# ==========================================

(
    silver_flight_current_df
    .write
    .format("delta")
    .mode("overwrite")
    .option("overwriteSchema", "true")
    .saveAsTable(SILVER_FLIGHT_CURRENT)
)

saved_current_df = spark.table(SILVER_FLIGHT_CURRENT)

print("Current status table:", SILVER_FLIGHT_CURRENT)
print("Rows saved:", saved_current_df.count())

display(
    saved_current_df
    .select(
        "flight_id",
        "source_sequence",
        "change_timestamp",
        "new_status"
    )
    .orderBy("flight_id")
    .limit(10)
)

In [0]:
# ==========================================
# TASK 6.7 - RECONCILE FLIGHTS AND CDC
# ==========================================

silver_flights = spark.table(SILVER_FLIGHT_TABLE)
silver_current = spark.table(SILVER_FLIGHT_CURRENT)

# Find CDC flights that match Silver flight events
matched_flights = (
    silver_current
    .join(
        silver_flights.select("flight_id").distinct(),
        on="flight_id",
        how="inner"
    )
)

# Find CDC flights with no matching Silver flight event
unmatched_cdc_flights = (
    silver_current
    .join(
        silver_flights.select("flight_id").distinct(),
        on="flight_id",
        how="left_anti"
    )
)

print("Distinct Silver flight IDs:",
      silver_flights.select("flight_id").distinct().count())

print("CDC current-status records:",
      silver_current.count())

print("Matched CDC flights:", matched_flights.count())

print("Unmatched CDC flights:", unmatched_cdc_flights.count())

In [0]:
# ==========================================
# TASK 6.8 - ENRICH SILVER FLIGHT EVENTS
# ==========================================

silver_flights = spark.table(SILVER_FLIGHT_TABLE)
silver_current = spark.table(SILVER_FLIGHT_CURRENT)

silver_enriched_df = (
    silver_flights.alias("f")
    .join(
        silver_current.select(
            "flight_id",
            F.col("new_status").alias("cdc_current_status"),
            F.col("source_sequence").alias("cdc_source_sequence"),
            F.col("change_timestamp").alias("cdc_change_timestamp")
        ).alias("c"),
        on="flight_id",
        how="left"
    )
)

print("Enriched Silver records:", silver_enriched_df.count())

print(
    "Records with CDC status:",
    silver_enriched_df
    .filter(F.col("cdc_current_status").isNotNull())
    .count()
)

print(
    "Records without CDC status:",
    silver_enriched_df
    .filter(F.col("cdc_current_status").isNull())
    .count()
)

display(
    silver_enriched_df
    .select(
        "flight_id",
        "flight_status",
        "cdc_current_status",
        "cdc_source_sequence",
        "cdc_change_timestamp"
    )
    .limit(15)
)

In [0]:
# ==========================================
# TASK 7.1 - INSPECT SECONDARY EVENTS
# ==========================================

secondary_df = spark.table(BRONZE_SECONDARY_TABLE)

print("Total secondary events:", secondary_df.count())

secondary_df.printSchema()

display(
    secondary_df
    .select(
        "event_id",
        "flight_id",
        "event_timestamp",
        "related_entity_id",
        "event_type",
        "event_value",
        "source_system"
    )
    .limit(15)
)

In [0]:
# ==========================================
# TASK 7.2 - PROFILE SECONDARY EVENTS
# ==========================================

secondary_df.select(
    F.count("*").alias("total_records"),
    F.sum(F.col("event_id").isNull().cast("int")).alias("null_event_id"),
    F.sum(F.col("flight_id").isNull().cast("int")).alias("null_flight_id"),
    F.sum(F.col("event_timestamp").isNull().cast("int")).alias("null_event_timestamp"),
    F.sum(F.col("event_type").isNull().cast("int")).alias("null_event_type"),
    F.sum(F.col("event_value").isNull().cast("int")).alias("null_event_value")
).show(truncate=False)

print(
    "Distinct event IDs:",
    secondary_df.select("event_id").distinct().count()
)

print(
    "Distinct flight IDs:",
    secondary_df.select("flight_id").distinct().count()
)

print("Event types:")
display(
    secondary_df
    .groupBy("event_type")
    .count()
    .orderBy("event_type")
)

# Check whether secondary events link to Silver flights
unmatched_secondary = (
    secondary_df
    .join(
        silver_flights.select("flight_id").distinct(),
        on="flight_id",
        how="left_anti"
    )
)

print("Unmatched secondary flight IDs:", unmatched_secondary.count())

In [0]:
# Event type distribution
display(
    secondary_df
    .groupBy("event_type")
    .count()
    .orderBy("event_type")
)

# Check secondary events against Silver flights
unmatched_secondary = (
    secondary_df
    .join(
        silver_flights.select("flight_id").distinct(),
        on="flight_id",
        how="left_anti"
    )
)

print("Unmatched secondary flight IDs:", unmatched_secondary.count())

In [0]:
# ==========================================
# TASK 7.3 - PREPARE SECONDARY EVENTS
# ==========================================

silver_secondary_df = (
    secondary_df
    .withColumn("event_id", F.trim(F.col("event_id")))
    .withColumn("flight_id", F.trim(F.col("flight_id")))
    .withColumn("related_entity_id", F.trim(F.col("related_entity_id")))
    .withColumn("event_type", F.upper(F.trim(F.col("event_type"))))
    .withColumn("source_system", F.upper(F.trim(F.col("source_system"))))
    .withColumn("event_timestamp", F.col("event_timestamp").cast("timestamp"))
    .withColumn("event_value", F.col("event_value").cast("double"))
)

print("Prepared secondary records:", silver_secondary_df.count())

# Save the secondary Silver table
(
    silver_secondary_df
    .write
    .format("delta")
    .mode("overwrite")
    .option("overwriteSchema", "true")
    .saveAsTable(
        f"{CATALOG}.{SILVER_SCHEMA}.silver_secondary_events"
    )
)

saved_secondary_df = spark.table(
    f"{CATALOG}.{SILVER_SCHEMA}.silver_secondary_events"
)

print("Silver secondary table created.")
print("Rows saved:", saved_secondary_df.count())

In [0]:
# ==========================================
# TASK 7.4 - SILVER TABLE RECONCILIATION
# ==========================================

silver_tables = [
    "silver_flight_events",
    "silver_rejected_records",
    "silver_status_history",
    "silver_flight_current",
    "silver_secondary_events"
]

for table_name in silver_tables:
    full_table_name = f"{CATALOG}.{SILVER_SCHEMA}.{table_name}"

    count = spark.table(full_table_name).count()

    print(f"{table_name}: {count} rows")

In [0]:
# ==========================================
# TASK 8.1 - SILVER DATA QUALITY TESTS
# ==========================================

flight_df = spark.table(SILVER_FLIGHT_TABLE)
rejected_df = spark.table(SILVER_REJECTED_TABLE)
history_df = spark.table(SILVER_STATUS_HISTORY)
current_df = spark.table(SILVER_FLIGHT_CURRENT)
secondary_silver_df = spark.table(
    f"{CATALOG}.{SILVER_SCHEMA}.silver_secondary_events"
)

tests = []

# Test 1: Flight event count
tests.append((
    "Silver flight events count",
    flight_df.count() == 11500
))

# Test 2: No duplicate exact source records
tests.append((
    "No exact duplicate source records",
    flight_df.select(*source_columns).distinct().count()
    == flight_df.count()
))

# Test 3: Rejected table exists and is empty
tests.append((
    "Rejected records table is empty",
    rejected_df.count() == 0
))

# Test 4: Status history count
tests.append((
    "Status history count",
    history_df.count() == 520
))

# Test 5: One current status per flight
tests.append((
    "Current status flight IDs are unique",
    current_df.select("flight_id").distinct().count()
    == current_df.count()
))

# Test 6: Secondary event IDs are unique
tests.append((
    "Secondary event IDs are unique",
    secondary_silver_df.select("event_id").distinct().count()
    == secondary_silver_df.count()
))

for test_name, passed in tests:
    print(("PASS" if passed else "FAIL") + " - " + test_name)

passed_count = sum(1 for _, passed in tests if passed)

print(f"\nTests passed: {passed_count}/{len(tests)}")

In [0]:
# ==========================================
# TASK 8.2 - TEST LATE-ARRIVING CDC
# ==========================================

test_cdc_df = spark.createDataFrame(
    [
        ("TEST-FLT-001", "2026-11-03 09:00:00",
         "PENDING", "COMPLETED", 101,
         "2026-11-03 09:01:00"),

        ("TEST-FLT-001", "2026-11-03 08:55:00",
         "PENDING", "OPEN", 100,
         "2026-11-03 09:10:00")
    ],
    """
    flight_id STRING,
    change_timestamp STRING,
    previous_status STRING,
    new_status STRING,
    source_sequence INT,
    arrival_timestamp STRING
    """
)

test_cdc_df = (
    test_cdc_df
    .withColumn(
        "change_timestamp",
        F.to_timestamp("change_timestamp")
    )
    .withColumn(
        "arrival_timestamp",
        F.to_timestamp("arrival_timestamp")
    )
)

# Select the highest source sequence, not the latest arrival
test_window = (
    Window
    .partitionBy("flight_id")
    .orderBy(
        F.col("source_sequence").desc(),
        F.col("change_timestamp").desc()
    )
)

test_current_df = (
    test_cdc_df
    .withColumn("_rank", F.row_number().over(test_window))
    .filter(F.col("_rank") == 1)
)

result = test_current_df.first()

print("Test CDC records:", test_cdc_df.count())
print("Selected sequence:", result["source_sequence"])
print("Selected current status:", result["new_status"])
print(
    "Late-arriving older update preserved in history:",
    test_cdc_df.count() == 2
)

assert result["source_sequence"] == 101
assert result["new_status"] == "COMPLETED"
assert test_cdc_df.count() == 2

print("PASS - Older late-arriving update did not replace newer status")

In [0]:
# ==========================================
# TASK 8.3 - TEST INVALID RECORD QUARANTINE
# ==========================================

test_invalid_df = spark.createDataFrame(
    [
        ("TEST-FLT-002", "COMPLETED", 500.0),
        (None, "OPEN", 300.0)
    ],
    """
    flight_id STRING,
    flight_status STRING,
    fare_amount DOUBLE
    """
)

test_invalid_result_df = (
    test_invalid_df
    .withColumn(
        "dq_reason",
        F.when(
            F.col("flight_id").isNull() |
            (F.trim(F.col("flight_id")) == ""),
            "DQ01: Missing flight_id"
        ).otherwise("")
    )
    .withColumn(
        "dq_status",
        F.when(F.col("dq_reason") != "", "REJECTED")
         .otherwise("VALID")
    )
)

valid_test_count = (
    test_invalid_result_df
    .filter(F.col("dq_status") == "VALID")
    .count()
)

rejected_test_count = (
    test_invalid_result_df
    .filter(F.col("dq_status") == "REJECTED")
    .count()
)

print("Test input records:", test_invalid_df.count())
print("Valid test records:", valid_test_count)
print("Rejected test records:", rejected_test_count)

display(test_invalid_result_df)

assert valid_test_count == 1
assert rejected_test_count == 1

print("PASS - Invalid mandatory field correctly rejected")

In [0]:
# ==========================================
# TASK 8.4 - VERIFY DQ RULE CATALOGUE
# ==========================================
CATALOG = "capstone_3flight"
BRONZE_SCHEMA = "flight_schema"
SILVER_SCHEMA = "silver_schema"

DQ_TABLE = f"{CATALOG}.{SILVER_SCHEMA}.dq_rule_catalogue"

dq_catalogue_df = spark.table(DQ_TABLE)

print("DQ catalogue table:", DQ_TABLE)
print("Total DQ rules:", dq_catalogue_df.count())

display(
    dq_catalogue_df.orderBy("rule_id")
)

# Day 2 – Silver Layer Limitations

1. **Timezone conversion:** The primary flight source contains a `timezone` column, but its values are `USD`, not valid timezone identifiers. Reliable local-to-UTC conversion could not be implemented.

2. **Schedule revisions:** No separate schedule-revision dataset was supplied. Matching flight events to schedule revisions by effective timestamp could not be implemented.

3. **CDC late-arrival testing:** A synthetic test confirmed that selecting the highest `source_sequence` prevents an older late-arriving update from replacing a newer status. The production CDC Delta MERGE and incremental recovery process has not yet been implemented or tested.

4. **CDC source coverage:** The supplied 520 CDC records relate to 520 distinct flights. Multiple updates for the same flight were not present in the supplied CDC batch.

5. **Quarantine testing:** The production rejected-records table was created with zero rows. A separate synthetic test verified the missing-flight-ID rejection rule, but no invalid record was inserted into the production quarantine table.

6. **Extreme fare threshold:** The source profile identified 35 fares above 5,000. This is treated as a data-quality investigation flag, not a confirmed business-approved fare limit.

In [0]:

# ============================================================
# CAPSTONE 03 - DAY 2 FINAL VALIDATION
# Read-only validation of Bronze, Silver, DQ, CDC and tests
# ============================================================

from pyspark.sql import functions as F
from pyspark.sql.window import Window

# ------------------------------------------------------------
# 1. CONFIGURATION
# ------------------------------------------------------------

CATALOG = "capstone_3flight"
BRONZE_SCHEMA = "flight_schema"
SILVER_SCHEMA = "silver_schema"

def table_name(schema, name):
    return f"{CATALOG}.{schema}.{name}"

BRONZE_FLIGHT = table_name(BRONZE_SCHEMA, "bronze_flight_events")
BRONZE_CDC = table_name(BRONZE_SCHEMA, "bronze_status_cdc")
BRONZE_SECONDARY = table_name(BRONZE_SCHEMA, "bronze_secondary_events")
AIRLINE = table_name(BRONZE_SCHEMA, "airline")
AIRPORT = table_name(BRONZE_SCHEMA, "airport")

SILVER_FLIGHT = table_name(SILVER_SCHEMA, "silver_flight_events")
SILVER_REJECTED = table_name(SILVER_SCHEMA, "silver_rejected_records")
SILVER_HISTORY = table_name(SILVER_SCHEMA, "silver_status_history")
SILVER_CURRENT = table_name(SILVER_SCHEMA, "silver_flight_current")
SILVER_SECONDARY = table_name(SILVER_SCHEMA, "silver_secondary_events")
DQ_CATALOGUE = table_name(SILVER_SCHEMA, "dq_rule_catalogue")

# ------------------------------------------------------------
# 2. VALIDATION FRAMEWORK
# ------------------------------------------------------------

results = []

def check(name, condition, detail=""):
    status = "PASS" if condition else "FAIL"
    results.append((name, status, detail))
    print(f"{status} | {name}" + (f" | {detail}" if detail else ""))

def info(name, detail):
    results.append((name, "INFO", detail))
    print(f"INFO | {name} | {detail}")

# ------------------------------------------------------------
# 3. LOAD TABLES
# ------------------------------------------------------------

table_names = [
    BRONZE_FLIGHT, BRONZE_CDC, BRONZE_SECONDARY,
    AIRLINE, AIRPORT,
    SILVER_FLIGHT, SILVER_REJECTED, SILVER_HISTORY,
    SILVER_CURRENT, SILVER_SECONDARY, DQ_CATALOGUE
]

for name in table_names:
    try:
        exists = spark.catalog.tableExists(name)
        check("Table exists: " + name, exists)
    except Exception as e:
        check("Table exists: " + name, False, str(e))

# Load data after checking table existence
bronze_flight = spark.table(BRONZE_FLIGHT)
bronze_cdc = spark.table(BRONZE_CDC)
bronze_secondary = spark.table(BRONZE_SECONDARY)
airline = spark.table(AIRLINE)
airport = spark.table(AIRPORT)

flight = spark.table(SILVER_FLIGHT)
rejected = spark.table(SILVER_REJECTED)
history = spark.table(SILVER_HISTORY)
current = spark.table(SILVER_CURRENT)
secondary = spark.table(SILVER_SECONDARY)
dq = spark.table(DQ_CATALOGUE)

# ------------------------------------------------------------
# 4. BRONZE SOURCE COUNT VALIDATION
# ------------------------------------------------------------

check("Bronze flight count", bronze_flight.count() == 11540,
      f"Actual: {bronze_flight.count()}")

check("Bronze CDC count", bronze_cdc.count() == 520,
      f"Actual: {bronze_cdc.count()}")

check("Bronze secondary count", bronze_secondary.count() == 310,
      f"Actual: {bronze_secondary.count()}")

check("Airline reference count", airline.count() == 110,
      f"Actual: {airline.count()}")

check("Airport reference count", airport.count() == 150,
      f"Actual: {airport.count()}")

# ------------------------------------------------------------
# 5. SILVER TABLE COUNT VALIDATION
# ------------------------------------------------------------

check("Silver flight count", flight.count() == 11500,
      f"Actual: {flight.count()}")

check("Rejected records count", rejected.count() == 0,
      f"Actual: {rejected.count()}")

check("Status history count", history.count() == 520,
      f"Actual: {history.count()}")

check("Current status count", current.count() == 520,
      f"Actual: {current.count()}")

check("Secondary events count", secondary.count() == 310,
      f"Actual: {secondary.count()}")

check("DQ catalogue count", dq.count() == 8,
      f"Actual: {dq.count()}")

# ------------------------------------------------------------
# 6. FLIGHT DATA QUALITY VALIDATION
# ------------------------------------------------------------

source_columns = [
    "flight_id", "event_timestamp", "ingest_timestamp",
    "airline_id", "airport_id", "flight_type",
    "flight_status", "timezone", "fare_amount",
    "delay_minutes", "aircraft_id", "source_system"
]

duplicate_count = (
    flight.count() -
    flight.select(*source_columns).distinct().count()
)

check("No exact source duplicates in Silver",
      duplicate_count == 0,
      f"Duplicate rows: {duplicate_count}")

missing_flight_type = flight.filter(
    F.col("flight_type").isNull()
).count()

check("Optional flight_type values preserved",
      missing_flight_type == 12,
      f"Missing flight_type: {missing_flight_type}")

null_flight_ids = flight.filter(
    F.col("flight_id").isNull()
).count()

check("No null flight IDs in Silver",
      null_flight_ids == 0,
      f"Null IDs: {null_flight_ids}")

# Reference integrity
airline_keys = airline.select("airline_id").distinct()
airport_keys = airport.select("airport_id").distinct()

unmatched_airlines = flight.join(
    airline_keys, "airline_id", "left_anti"
).count()

unmatched_airports = flight.join(
    airport_keys, "airport_id", "left_anti"
).count()

check("Airline reference integrity",
      unmatched_airlines == 0,
      f"Unmatched: {unmatched_airlines}")

check("Airport reference integrity",
      unmatched_airports == 0,
      f"Unmatched: {unmatched_airports}")

# Bronze ingestion metadata
rescued_rows = bronze_flight.filter(
    F.col("_rescued_data").isNotNull()
).count()

missing_metadata = bronze_flight.filter(
    F.col("_source_file").isNull() |
    F.col("_ingested_at").isNull()
).count()

check("No rescued Bronze records",
      rescued_rows == 0,
      f"Rescued: {rescued_rows}")

check("Bronze ingestion metadata populated",
      missing_metadata == 0,
      f"Missing metadata: {missing_metadata}")

# Source anomaly profile
extreme_fares = bronze_flight.filter(
    F.col("fare_amount") > 5000
).count()

check("Injected extreme fare profile",
      extreme_fares == 35,
      f"Bronze fares above 5000: {extreme_fares}")

# ------------------------------------------------------------
# 7. CDC VALIDATION
# ------------------------------------------------------------

cdc_nulls = bronze_cdc.filter(
    F.col("flight_id").isNull() |
    F.col("change_timestamp").isNull() |
    F.col("source_sequence").isNull() |
    F.col("new_status").isNull()
).count()

check("CDC mandatory fields populated",
      cdc_nulls == 0,
      f"Invalid records: {cdc_nulls}")

sequence_stats = bronze_cdc.agg(
    F.min("source_sequence").alias("min_seq"),
    F.max("source_sequence").alias("max_seq"),
    F.countDistinct("source_sequence").alias("distinct_seq")
).first()

check("CDC sequence range",
      sequence_stats["min_seq"] == 1 and
      sequence_stats["max_seq"] == 520 and
      sequence_stats["distinct_seq"] == 520,
      f"Min={sequence_stats['min_seq']}, "
      f"Max={sequence_stats['max_seq']}, "
      f"Distinct={sequence_stats['distinct_seq']}")

repeated_cdc_sequences = (
    bronze_cdc
    .groupBy("flight_id", "source_sequence")
    .count()
    .filter(F.col("count") > 1)
    .count()
)

check("No repeated flight/sequence combinations",
      repeated_cdc_sequences == 0,
      f"Repeated groups: {repeated_cdc_sequences}")

# All Bronze CDC events retained in history
missing_history = (
    bronze_cdc.select("flight_id", "source_sequence")
    .join(
        history.select("flight_id", "source_sequence"),
        ["flight_id", "source_sequence"],
        "left_anti"
    )
    .count()
)

check("All CDC events retained in history",
      missing_history == 0,
      f"Missing history events: {missing_history}")

# Validate latest sequence selected for each flight
latest_per_flight = (
    history
    .groupBy("flight_id")
    .agg(F.max("source_sequence").alias("expected_sequence"))
)

incorrect_current = (
    current.alias("c")
    .join(latest_per_flight.alias("h"), "flight_id")
    .filter(
        F.col("c.source_sequence") !=
        F.col("h.expected_sequence")
    )
    .count()
)

check("Current table uses latest sequence",
      incorrect_current == 0,
      f"Incorrect current rows: {incorrect_current}")

check("Current flight IDs are unique",
      current.select("flight_id").distinct().count()
      == current.count())

# CDC-to-flight relationship
unmatched_cdc = (
    current.select("flight_id").distinct()
    .join(
        flight.select("flight_id").distinct(),
        "flight_id",
        "left_anti"
    )
    .count()
)

check("CDC flights match Silver flights",
      unmatched_cdc == 0,
      f"Unmatched: {unmatched_cdc}")

# ------------------------------------------------------------
# 8. SECONDARY EVENT VALIDATION
# ------------------------------------------------------------

secondary_duplicates = (
    secondary.count() -
    secondary.select("event_id").distinct().count()
)

check("Secondary event IDs unique",
      secondary_duplicates == 0,
      f"Duplicate IDs: {secondary_duplicates}")

unmatched_secondary = (
    secondary.select("flight_id").distinct()
    .join(
        flight.select("flight_id").distinct(),
        "flight_id",
        "left_anti"
    )
    .count()
)

check("Secondary events match Silver flights",
      unmatched_secondary == 0,
      f"Unmatched flights: {unmatched_secondary}")

# ------------------------------------------------------------
# 9. LATE-ARRIVING CDC UNIT TEST
# ------------------------------------------------------------

test_cdc = spark.createDataFrame(
    [
        ("TEST-001", 101, "COMPLETED"),
        ("TEST-001", 100, "OPEN")
    ],
    ["flight_id", "source_sequence", "new_status"]
)

test_window = (
    Window.partitionBy("flight_id")
    .orderBy(F.col("source_sequence").desc())
)

test_latest = (
    test_cdc
    .withColumn("_rank", F.row_number().over(test_window))
    .filter(F.col("_rank") == 1)
    .first()
)

check("Late CDC update does not replace newer status",
      test_latest["source_sequence"] == 101 and
      test_latest["new_status"] == "COMPLETED",
      f"Selected sequence: {test_latest['source_sequence']}")

# ------------------------------------------------------------
# 10. INVALID RECORD QUARANTINE UNIT TEST
# ------------------------------------------------------------

test_invalid = spark.createDataFrame(
    [
        ("TEST-VALID", "COMPLETED"),
        (None, "OPEN")
    ],
    ["flight_id", "flight_status"]
)

test_invalid_result = test_invalid.withColumn(
    "dq_status",
    F.when(
        F.col("flight_id").isNull() |
        (F.trim(F.col("flight_id")) == ""),
        "REJECTED"
    ).otherwise("VALID")
)

test_valid_count = test_invalid_result.filter(
    F.col("dq_status") == "VALID"
).count()

test_rejected_count = test_invalid_result.filter(
    F.col("dq_status") == "REJECTED"
).count()

check("Invalid flight ID quarantine unit test",
      test_valid_count == 1 and test_rejected_count == 1,
      f"Valid={test_valid_count}, Rejected={test_rejected_count}")

# ------------------------------------------------------------
# 11. PROJECT LIMITATIONS
# ------------------------------------------------------------

info(
    "Timezone conversion",
    "Not verified: source timezone values were profiled as USD; "
    "valid timezone identifiers are unavailable."
)

info(
    "Schedule revision matching",
    "Not implemented: no separate schedule-revision dataset supplied."
)

info(
    "Production incremental CDC merge",
    "Not verified: synthetic sequence-selection test passed, "
    "but production Delta MERGE/recovery was not tested."
)

info(
    "Production quarantine with actual bad row",
    "Not demonstrated: production quarantine table is empty; "
    "synthetic invalid-record test passed."
)

# ------------------------------------------------------------
# 12. FINAL SUMMARY
# ------------------------------------------------------------

passed = sum(1 for r in results if r[1] == "PASS")
failed = sum(1 for r in results if r[1] == "FAIL")
informational = sum(1 for r in results if r[1] == "INFO")

print("\n" + "=" * 65)
print("DAY 2 FINAL VALIDATION SUMMARY")
print("=" * 65)
print("Passed:", passed)
print("Failed:", failed)
print("Information / limitations:", informational)
print("Total checks:", len(results))

if failed == 0:
    print("\nRESULT: ALL EXECUTED VALIDATION CHECKS PASSED")
else:
    print("\nRESULT: VALIDATION NEEDS ATTENTION")

print("=" * 65)

# Display failed checks, if any
if failed > 0:
    print("\nFAILED CHECKS:")
    for name, status, detail in results:
        if status == "FAIL":
            print("-", name, detail)
